# 04 — Một hàng đầu vào và một mục tiêu

Sau khi đóng cửa phiên `t`, ta tính các đặc trưng từ phiên đó và quá khứ để dự báo phiên giao dịch quan sát được kế tiếp `t+1`. Không chèn ngày nghỉ vào chuỗi. Dữ liệu dừng tại **30/09/2025**.

Danh mục đủ 32 biến, công thức, mục đích và bài tính 5 phiên: [EDA và đặc trưng](../docs/EDA_VA_DAC_TRUNG.md).

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Từ 5 trường giao dịch thành 32 biến

Có **5 trường gốc** (open, high, low, close, volume), **23 chỉ báo kỹ thuật** và **4 đặc trưng nến**. Một chỉ báo là phép tóm tắt dữ liệu đã biết, không phải lời khẳng định phiên sau sẽ tăng/giảm.

| Nhóm | Hỏi dữ liệu điều gì? | Ví dụ công thức |
|---|---|---|
| Return 1/2/5/10/20 phiên | Giá đổi bao nhiêu trong khoảng nhìn lại? | `close_t / close_(t-k) − 1` |
| SMA/EMA ratio 5/10/20/50 | Giá đang lệch mức nền bao nhiêu? | `close / đường_trung_bình − 1` |
| RSI, MACD | Mức tăng/giảm và xu hướng nhanh/chậm ra sao? | RSI lưu ở thang 0–1; MACD chia cho close |
| Bollinger, volatility | Vùng dao động rộng hẹp thế nào? | Độ lệch chuẩn cửa sổ, chia tổng bình phương cho `k` (`ddof=0`) |
| Log-volume | Khối lượng bất thường so với lịch sử không? | Dùng `ln(1+volume)` để chấp nhận volume bằng 0 |
| Nến | Giá đi từ mở đến đóng và nằm ở đâu trong biên độ? | `(close-open)/open`; `(close-low)/(high-low)` |

**Lưu ý:** `return_5d` là thay đổi từ 5 phiên trước đến hiện tại, không phải lợi nhuận một ngày của 5 phiên trước. `next_log_return` và `next_close` là đáp án, không có trong 32 đầu vào.

SMA50 cần 50 giá quá khứ/hiện tại nên 49 hàng đầu chưa đủ chỉ báo. Bảng còn 4.620 hàng, nhưng hàng cuối chưa biết đáp án phiên sau. Ô dưới in hai hàng cuối để kiểm tra sự khác nhau này.

In [ ]:
from features import load_price_data,build_feature_table,PRICE_ONLY_FEATURE_COLUMNS
raw=load_price_data(ROOT/'data/raw/fpt/fpt_historical_price.csv')
table=build_feature_table(raw)
print('Số đầu vào:',len(PRICE_ONLY_FEATURE_COLUMNS))
display(table.iloc[[-2,-1]].T)

## Một dự báo sinh ra giá và hướng

Mục tiêu học là `r = ln(close_kế_tiếp / close_hiện_tại)`. Sau dự báo, đổi ngược: `giá_dự_báo = giá_hiện_tại × exp(r_dự_báo)`.

Ví dụ `ln(1.01) ≈ 0.00995`, nhân `100 × exp(0.00995)` được khoảng 101; dấu dương nghĩa là tăng. `ln(0.99)` âm nên giá giảm về 99. Giá trị đúng 0 nghĩa là đứng giá, không tự coi là tăng.

Trong hàng thật **29/09/2025**, close hiện tại là 84.75625 và close phiên mục tiêu 30/09 là 82.54: log return là `ln(82.54/84.75625) ≈ −0.02649647`, còn lợi nhuận đơn là khoảng −2.6149%. Hai đại lượng gần nhau nhưng không bằng nhau. Ô dưới minh họa phép quy đổi với các số tròn.

In [ ]:
import numpy as np
from metrics import restore_price,direction_from_return
example=np.log([1.01,0.99,1.0])
display(pd.DataFrame({'log_return':example,'predicted_close':restore_price([100,100,100],example),'direction':direction_from_return(example)}))

## Tự kiểm tra bằng 5 giá: 10, 12, 11, 14, 13

- Return từ 10 lên 12: `(12−10)/10 = 20%`.
- SMA3 cuối: `(11+14+13)/3 = 12.6667`; tỷ lệ giá/SMA là `13/12.6667−1 ≈ 2.6316%`.
- EMA3 với trọng số mới 0.5 lần lượt là `10 → 11 → 11 → 12.5 → 12.75`.
- Trung bình 5 giá là 12; bình phương các độ lệch cộng thành 10; chia 5 rồi lấy căn được độ lệch chuẩn `sqrt(2) ≈ 1.4142`.

Đây là chu kỳ nhỏ để học phép tính. Dự án thật dùng RSI14, MACD12/26/9, Bollinger20, SMA/EMA5/10/20/50. Bài tính chi tiết RSI, MACD, Bollinger, volatility, volume và nến nằm trong [EDA và đặc trưng](../docs/EDA_VA_DAC_TRUNG.md).

Bảng kỹ thuật có hàng đầu vào đầu tiên ngày 20/03/2007. Notebook 05 ghép nguồn bổ trợ và yêu cầu **60 phiên liên tiếp đủ 42 biến**. Hàng đầu tiên của bảng dùng chung có đầu vào ngày 06/08/2013 và nhãn ngày 07/08/2013.